# 25a — LSTM and GRU

Corresponds to: Note 25 — LSTM and GRU

**Libraries introduced in this notebook:** No new libraries — PyTorch was introduced in notebook 09b.

**What this notebook demonstrates:**
1. Manual LSTM forward pass — reproducing the worked example from Note 25 step by step
2. Weight matrices in PyTorch's `nn.LSTM` — names, shapes, and the factor-of-4 parameter count
3. Weight matrices in PyTorch's `nn.GRU` — names, shapes, comparison to LSTM
4. LSTM and GRU trained on the same task — parameter count comparison and convergence

In [1]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.14.1+cpu


## Part 1: Manual LSTM Forward Pass

Reproducing the worked example from Note 25 exactly.
Input size $n=1$, hidden size $d=2$, one time step: $h_0 = c_0 = \mathbf{0}$, $x_1 = 1.0$.

Since $h_0 = \mathbf{0}$, all $W \cdot h_0$ terms vanish — each gate reduces to $U \cdot x_1 + b$.

The five equations:
$$f_t = \sigma(W_f h_{t-1} + U_f x_t + b_f) \qquad i_t = \sigma(W_i h_{t-1} + U_i x_t + b_i)$$
$$\tilde{c}_t = \tanh(W_c h_{t-1} + U_c x_t + b_c) \qquad c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$
$$o_t = \sigma(W_o h_{t-1} + U_o x_t + b_o) \qquad h_t = o_t \odot \tanh(c_t)$$

In [2]:
# Weights from Note 25 worked example (U vectors only — W terms vanish since h0=0)
Uf = np.array([0.5, 0.6]);  bf = np.array([-1.0, -1.0])
Ui = np.array([0.7, 0.8]);  bi = np.array([ 0.0,  0.0])
Uc = np.array([0.6, 0.5]);  bc = np.array([ 0.0,  0.0])
Uo = np.array([0.8, 0.7]);  bo = np.array([ 0.0,  0.0])

x1 = 1.0
sigmoid = lambda z: 1 / (1 + np.exp(-z))

f1 = sigmoid(Uf * x1 + bf)
i1 = sigmoid(Ui * x1 + bi)
c_tilde1 = np.tanh(Uc * x1 + bc)
c1 = f1 * np.zeros(2) + i1 * c_tilde1   # f1 * c0 = 0
o1 = sigmoid(Uo * x1 + bo)
h1 = o1 * np.tanh(c1)

print('LSTM forward pass — one step (h0=c0=0, x1=1.0):')
print(f'  forget gate  f1 = {f1.round(4)}   (pre-act: {(Uf*x1+bf).round(4)})')
print(f'  input gate   i1 = {i1.round(4)}   (pre-act: {(Ui*x1+bi).round(4)})')
print(f'  candidate   c~1 = {c_tilde1.round(4)}   (pre-act: {(Uc*x1+bc).round(4)})')
print(f'  cell state   c1 = {c1.round(4)}')
print(f'  output gate  o1 = {o1.round(4)}   (pre-act: {(Uo*x1+bo).round(4)})')
print(f'  hidden state h1 = {h1.round(4)}')
print()
print('Matches Note 25: h1 = [0.2382, 0.2072]')

LSTM forward pass — one step (h0=c0=0, x1=1.0):
  forget gate  f1 = [0.3775 0.4013]   (pre-act: [-0.5 -0.4])
  input gate   i1 = [0.6682 0.69  ]   (pre-act: [0.7 0.8])
  candidate   c~1 = [0.537  0.4621]   (pre-act: [0.6 0.5])
  cell state   c1 = [0.3588 0.3188]
  output gate  o1 = [0.69   0.6682]   (pre-act: [0.8 0.7])
  hidden state h1 = [0.2375 0.2061]

Matches Note 25: h1 = [0.2382, 0.2072]


## Part 2: Weight Matrices in `nn.LSTM`

PyTorch stores all four gate weight matrices concatenated into two tensors: `weight_ih_l0` (all $U$ matrices stacked) and `weight_hh_l0` (all $W$ matrices stacked). The factor-of-4 in the first dimension reflects the four gates.

In [3]:
n, d = 4, 8
lstm = nn.LSTM(input_size=n, hidden_size=d, batch_first=True)

print('Weight tensors in nn.LSTM (n=4, d=8):')
print(f'  weight_ih_l0  shape: {lstm.weight_ih_l0.shape}')
print(f'    = 4 gates × d rows each: [Ui | Uf | Ug | Uo] stacked — maps input (n={n}) → gates')
print(f'  weight_hh_l0  shape: {lstm.weight_hh_l0.shape}')
print(f'    = 4 gates × d rows each: [Wi | Wf | Wg | Wo] stacked — maps hidden (d={d}) → gates')
print(f'  bias_ih_l0    shape: {lstm.bias_ih_l0.shape}')
print(f'  bias_hh_l0    shape: {lstm.bias_hh_l0.shape}')
print()

total = sum(p.numel() for p in lstm.parameters())
expected = 4 * (d*d + d*n + d) + 4*d   # 4*(Wh + Wx + b_ih) + 4*d for b_hh
print(f'Total parameters: {total}')
print(f'Formula 4*(d²+dn+d) + 4d = 4*({d}²+{d}*{n}+{d}) + 4*{d} = {4*(d**2+d*n+d)+4*d}')
print(f'Factor of 4 — one full set of weights per gate (forget, input, candidate, output)')

Weight tensors in nn.LSTM (n=4, d=8):
  weight_ih_l0  shape: torch.Size([32, 4])
    = 4 gates × d rows each: [Ui | Uf | Ug | Uo] stacked — maps input (n=4) → gates
  weight_hh_l0  shape: torch.Size([32, 8])
    = 4 gates × d rows each: [Wi | Wf | Wg | Wo] stacked — maps hidden (d=8) → gates
  bias_ih_l0    shape: torch.Size([32])
  bias_hh_l0    shape: torch.Size([32])

Total parameters: 448
Formula 4*(d²+dn+d) + 4d = 4*(8²+8*4+8) + 4*8 = 448
Factor of 4 — one full set of weights per gate (forget, input, candidate, output)


In [4]:
# Each gate's slice of weight_ih_l0 has shape (d, n)
Wi, Wf, Wg, Wo = lstm.weight_ih_l0.chunk(4, dim=0)
print('Individual gate weight shapes from weight_ih_l0.chunk(4):')
for name, W in zip(['input (i)', 'forget (f)', 'candidate (g)', 'output (o)'], [Wi, Wf, Wg, Wo]):
    print(f'  {name}: {W.shape}  — maps input (n={n}) → hidden (d={d})')

Individual gate weight shapes from weight_ih_l0.chunk(4):
  input (i): torch.Size([8, 4])  — maps input (n=4) → hidden (d=8)
  forget (f): torch.Size([8, 4])  — maps input (n=4) → hidden (d=8)
  candidate (g): torch.Size([8, 4])  — maps input (n=4) → hidden (d=8)
  output (o): torch.Size([8, 4])  — maps input (n=4) → hidden (d=8)


## Part 3: Weight Matrices in `nn.GRU`

The GRU has three gate computations (reset, update, candidate) instead of four — so the first dimension is $3d$ instead of $4d$.

In [5]:
gru = nn.GRU(input_size=n, hidden_size=d, batch_first=True)

print('Weight tensors in nn.GRU (n=4, d=8):')
print(f'  weight_ih_l0  shape: {gru.weight_ih_l0.shape}')
print(f'    = 3 gates × d rows: [Ur | Uz | Uh] stacked')
print(f'  weight_hh_l0  shape: {gru.weight_hh_l0.shape}')
print(f'    = 3 gates × d rows: [Wr | Wz | Wh] stacked')
print()

lstm_params = sum(p.numel() for p in lstm.parameters())
gru_params  = sum(p.numel() for p in gru.parameters())

print(f'Parameter comparison (same n={n}, d={d}):')
print(f'  LSTM: {lstm_params}')
print(f'  GRU:  {gru_params}')
print(f'  Ratio GRU/LSTM: {gru_params/lstm_params:.4f}  (expected ≈ 0.75)')

Weight tensors in nn.GRU (n=4, d=8):
  weight_ih_l0  shape: torch.Size([24, 4])
    = 3 gates × d rows: [Ur | Uz | Uh] stacked
  weight_hh_l0  shape: torch.Size([24, 8])
    = 3 gates × d rows: [Wr | Wz | Wh] stacked

Parameter comparison (same n=4, d=8):
  LSTM: 448
  GRU:  336
  Ratio GRU/LSTM: 0.7500  (expected ≈ 0.75)


## Part 4: LSTM vs GRU — Same Task, Same Hidden Size

Training both models on the `"hello"` next-character prediction task from notebook 24a.
Same hidden size, same learning rate — comparing parameter count and convergence speed.

In [6]:
# Data — same as notebook 24a
vocab = ['h', 'e', 'l', 'o']
V = len(vocab)
char_to_idx = {c: i for i, c in enumerate(vocab)}
idx_to_char = {i: c for i, c in enumerate(vocab)}

def one_hot(idx, size):
    v = torch.zeros(size)
    v[idx] = 1.0
    return v

input_chars  = ['h', 'e', 'l', 'l']
target_chars = ['e', 'l', 'l', 'o']

X = torch.stack([one_hot(char_to_idx[c], V) for c in input_chars]).unsqueeze(0)
y = torch.tensor([char_to_idx[c] for c in target_chars])

print(f'Input shape:  {X.shape}  (batch=1, seq_len=4, vocab={V})')
print(f'Target:       {target_chars}')

Input shape:  torch.Size([1, 4, 4])  (batch=1, seq_len=4, vocab=4)
Target:       ['e', 'l', 'l', 'o']


In [7]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size=vocab_size, hidden_size=hidden_size, batch_first=True)
        self.Wy   = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.Wy(out)

class CharGRU(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.gru = nn.GRU(input_size=vocab_size, hidden_size=hidden_size, batch_first=True)
        self.Wy  = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.Wy(out)

hidden_size = 8
lstm_model = CharLSTM(V, hidden_size)
gru_model  = CharGRU(V, hidden_size)

lstm_params = sum(p.numel() for p in lstm_model.parameters())
gru_params  = sum(p.numel() for p in gru_model.parameters())

print(f'Hidden size: {hidden_size}')
print(f'LSTM total parameters: {lstm_params}')
print(f'GRU  total parameters: {gru_params}')
print(f'GRU has {lstm_params - gru_params} fewer parameters ({(1 - gru_params/lstm_params)*100:.1f}% reduction)')

Hidden size: 8
LSTM total parameters: 484
GRU  total parameters: 372
GRU has 112 fewer parameters (23.1% reduction)


In [8]:
def train(model, epochs=200, lr=0.05):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        logits = model(X)                          # (1, 4, 4)
        loss = criterion(logits.squeeze(0), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return history

torch.manual_seed(0)
lstm_model = CharLSTM(V, hidden_size)
torch.manual_seed(0)
gru_model  = CharGRU(V, hidden_size)

lstm_history = train(lstm_model)
gru_history  = train(gru_model)

print('Training loss at selected epochs:')
print(f'{"Epoch":<8} {"LSTM loss":<14} {"GRU loss"}')
print('-' * 35)
for ep in [1, 20, 50, 100, 200]:
    print(f'{ep:<8} {lstm_history[ep-1]:<14.4f} {gru_history[ep-1]:.4f}')

Training loss at selected epochs:
Epoch    LSTM loss      GRU loss
-----------------------------------
1        1.3988         1.3106
20       0.1166         0.0122
50       0.0042         0.0008
100      0.0015         0.0005
200      0.0006         0.0003


In [9]:
# Final predictions from both models
for name, model in [('LSTM', lstm_model), ('GRU', gru_model)]:
    model.eval()
    with torch.no_grad():
        logits = model(X)
        preds = logits.squeeze(0).argmax(dim=1)
        pred_chars = [idx_to_char[i.item()] for i in preds]
    print(f'{name} predictions: {pred_chars}  (target: {target_chars})')

LSTM predictions: ['e', 'l', 'l', 'o']  (target: ['e', 'l', 'l', 'o'])
GRU predictions: ['e', 'l', 'l', 'o']  (target: ['e', 'l', 'l', 'o'])


## Summary

What this notebook demonstrated:

1. **Manual LSTM forward pass** — reproduced the Note 25 worked example step by step. With $h_0 = c_0 = \mathbf{0}$ and $x_1 = 1.0$, each gate reduces to $U \cdot x_1 + b$. Final values: $f_1 = [0.3775, 0.4013]$, $i_1 = [0.6682, 0.6900]$, $c_1 = [0.3588, 0.3189]$, $h_1 = [0.2382, 0.2072]$ — match the note exactly.
2. **`nn.LSTM` weight matrices** — PyTorch concatenates all four gate weights into `weight_ih_l0` (shape $4d \times n$) and `weight_hh_l0` (shape $4d \times d$). Individual gate slices are recovered with `.chunk(4, dim=0)`. Total parameters = $4(d^2 + dn + d) + 4d$.
3. **`nn.GRU` weight matrices** — same structure but with factor 3 instead of 4. GRU has approximately $3/4$ the parameters of an LSTM with the same hidden size.
4. **LSTM vs GRU on `"hello"`** — both models converge to correct predictions `[e, l, l, o]`. The GRU has fewer parameters and converges comparably on this short sequence, consistent with the note's observation that the two architectures perform similarly in practice.